## **UPS**

**El error más común al comprar UPS es mirar solo los VA y no la batería.**

La autonomía depende de dos cosas únicamente: la capacidad de la batería en Wh (watt-hora) y el consumo de tu equipo.

            Autonomía (horas) = Wh de la batería / Watts de consumo

El problema es que los fabricantes no siempre muestran los Wh directamente, pero podés calcularlo vos mismo:

            Wh = Voltaje de batería × Ah

Pero en la práctica los UPS tienen pérdidas de eficiencia del 30–40%

**Las características que sí debes mirar:**



**1. Capacidad de batería en Ah**

Es el dato más importante para la autonomía. Mientras más Ah, más dura.

| Batería | Wh totales | Autonomía aprox a 120W|
|:-:|:-:|:-:|
|12V / 7Ah |84 Wh |~25 min|
|12V / 9Ah |108 Wh |~35 min|
|12V / 12Ah |144 Wh |~45 min|
|24V / 9Ah (2 baterías) |216 Wh |~80 min|
|24V / 12Ah (2 baterías) |288 Wh |~110 min|

Los UPS con dos baterías en serie de 24V total duran significativamente más.

**2. Tecnología**

línea interactiva o doble conversión

La línea interactiva es suficiente para tu caso. La doble conversión es más cara y para servidores críticos empresariales, overkill para vos.



**3. AVR incluido**

Imprescindible. Regula voltaje sin usar la batería, protegiéndola para cuando realmente la necesitás.



**4. Puerto USB para comunicación con el servidor**

Esto es importante. Con el software NUT en Ubuntu, el servidor detecta cuando el UPS está en batería y se apaga automáticamente antes de quedarse sin energía. 

Sin puerto USB tenés que apagarlo manualmente.



**5. Baterías reemplazables y conseguibles localmente**

Las baterías duran 2–3 años. Verificá que el modelo de batería que usa se consiga en tu localidad. Las baterías más comunes son las de 12V en tamaños 7Ah, 9Ah y 12Ah, que se consiguen en cualquier casa de electrónica.

**Lo que NO importa tanto:**

Los VA del UPS solo indican cuánta potencia eléctrica puede manejar simultáneamente, no cuánto dura. 

- Un UPS de 2000VA con batería chica dura menos que uno de 1000VA con batería grande.

El UPS ideal para el proyecto en términos de especificaciones:

- Tecnología: línea interactiva
- AVR: sí
- Potencia: 1000–1500 VA 
- Batería: 12V / 12Ah mínimo, mejor si tiene dos baterías
- Puerto USB: sí, imprescindible
- Baterías reemplazables: sí

**Conclusión**

- Cuando veas un UPS fijate primero en la batería (voltaje × Ah = Wh), luego verificá que tenga AVR y puerto USB. 

- Los VA son secundarios siempre que superen tu consumo máximo con margen. 

- Un UPS de 1000VA con batería de 12Ah te da mejor autonomía que uno de 2000VA con batería de 7Ah.

## **Que pasa con la informacion de los usuarios cuando el server se queda sin energia?**

- PostgreSQL como base de datos, que tiene un sistema de transacciones. 

- Esto significa que cada operación se completa entera o no se completa, nunca a medias. 

- Si se va la luz mientras un usuario está usando el sistema, lo peor que puede pasar es que se pierda el progreso de esos últimos segundos, pero los datos del usuario quedan intactos.

## **Los tres escenarios posibles:**

**Escenario 1 — Se va la luz mientras un usuario estaba usando el sistema**
- No pasa nada grave. El progreso guardado queda hasta el último punto que se registró. Puede perder 30 segundos o 1 minuto de progreso como máximo. Cuando vuelve la luz el usuario retoma desde donde quedó guardado.

**Escenario 2 — Se va la luz mientras se cargan archivos al servidor**
- El archivo que estabas subiendo puede quedar corrupto o incompleto. 
- El sistema lo detecta y lo marca como fallido. 
Simplemente lo subís de nuevo cuando vuelve la energia, sin afectar ningún otro archivo ni usuario.

**Escenario 3 — Se va la luz mientras la base de datos escribe datos críticos**
- Este es el caso más delicado. Si PostgreSQL estaba escribiendo en el disco exactamente en ese momento, el archivo de base de datos puede corromperse. Es poco frecuente pero posible después de muchos cortes bruscos repetidos.

## **¿Cómo protegerse?**

**Tres medidas que configuraremos desde el principio:**
- La primera es backups automáticos diarios de la base de datos. Un script que corre a las 3am todos los días y copia la base de datos al HDD de archivos. Si algo se corrompe, restaurás el backup del día anterior y perdés como máximo un día de actividad.
- La segunda es UPS básico. Como ya hablamos, aunque no es urgente para arrancar, un UPS de 650 VA da tiempo suficiente para que el servidor se apague ordenadamente cuando detecta corte de luz, evitando el escenario 3 completamente.
- La tercera es apagado automático por UPS. Con el software NUT en Ubuntu, cuando el UPS detecta batería baja manda una señal al servidor para que se apague limpiamente antes de quedarse sin energía.